# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [61]:
print("Hello World!")


Hello World!


In [62]:
import pandas as pd

# Path to the file the teacher gave you
file_path = 'loinc_dataset-v2.xlsx'

# 1. Load the three sheets (skipping the query header at the top)
df_glucose = pd.read_excel(file_path, sheet_name='glucose in blood', skiprows=2)
df_bilirubin = pd.read_excel(file_path, sheet_name='bilirubin in plasma', skiprows=2)
df_wbc = pd.read_excel(file_path, sheet_name='White blood cells count', skiprows=2)

# 2. Assign Query IDs (qid) and Relevance Labels
# We tell the model which LOINC code is the "Perfect Match" for each query
def prepare_data(df, qid, winner_code):
    df['qid'] = qid
    df['relevance'] = 0  # Default all to 'wrong'
    # Mark the correct LOINC code as '1' (Correct)
    df.loc[df['loinc_num'] == winner_code, 'relevance'] = 1
    return df

# Using the most accurate codes found in your data
df_glucose = prepare_data(df_glucose, qid=1, winner_code='74774-1') # Glucose Bld
df_bilirubin = prepare_data(df_bilirubin, qid=2, winner_code='1968-7') # Bilirubin Plas
df_wbc = prepare_data(df_wbc, qid=3, winner_code='26464-8') # WBC Count

# 3. Merge them into one Master Training Set
training_set = pd.concat([df_glucose, df_bilirubin, df_wbc], ignore_index=True)

print("Training set built successfully!")
print(training_set[['qid', 'loinc_num', 'relevance']].head())

# Λεξικό με τις ερωτήσεις για σύγκριση
query_texts = {
    1: "glucose in blood",
    2: "bilirubin in plasma",
    3: "white blood cells count"
}

queries = {1: "glucose in blood", 2: "bilirubin in plasma", 3: "white blood cells count"}

def extract_features(row):
    query = queries[row['qid']].lower()
    name = str(row['long_common_name']).lower()
    component = str(row['component']).lower()

    # Feature: Πόσες λέξεις της ερώτησης υπάρχουν στο όνομα;
    match_count = sum(1 for word in query.split() if word in name)

    # Component match
    comp_match = 1 if any(word in component for word in query.split()) else 0

    # White blood cells are shown as leukocytes
    # I have to use synonyms
    synonym_match = 0
    if "white blood cells count" in query and "leukocytes" in name:
        synonym_match = 1

    return pd.Series([match_count, comp_match, synonym_match])

training_set[['feature_match', 'feature_comp', 'synonym_match']] = training_set.apply(extract_features, axis=1)

Training set built successfully!
   qid loinc_num  relevance
0    1    1988-5          0
1    1    1959-6          0
2    1   10331-7          0
3    1   18998-5          0
4    1    1975-2          0


In [63]:
# METRICS
def calculate_metrics(df):
    # Mean Reciprocal Rank (MRR)
    def get_rr(group):
        # Sort by score and find the rank of the first relevance=1
        sorted_group = group.sort_values('score', ascending=False).reset_index()
        ranks = sorted_group.index[sorted_group['relevance'] == 1].tolist()
        return 1 / (ranks[0] + 1) if ranks else 0

    mrr = df.groupby('qid')[['score', 'relevance']].apply(get_rr, include_groups=False).mean()

    # Precision@5
    def get_p5(group):
        top_5 = group.sort_values('score', ascending=False).head(5)
        return 1 if top_5['relevance'].any() else 0

    p5 = df.groupby('qid')[['score', 'relevance']].apply(get_p5, include_groups=False).mean()

    return mrr, p5

# Υπολογισμός Precision@1
def calculate_p1(df):
    top_results = df.sort_values(by=['qid', 'score'], ascending=[True, False]).groupby('qid').head(1)
    precision_at_1 = top_results['relevance'].mean()
    return precision_at_1


In [64]:
from sklearn.ensemble import RandomForestClassifier

# X = τα χαρακτηριστικά, y = ο στόχος (σωστό/λάθος)
X = training_set[['feature_match', 'feature_comp', 'synonym_match']]
y = training_set['relevance']

model = RandomForestClassifier()
model.fit(X, y)

# Προβλέπουμε την πιθανότητα να είναι σωστό κάθε αποτέλεσμα
training_set['score'] = model.predict_proba(X)[:, 1]

In [65]:
# Ταξινομούμε το training_set ανά qid (ερώτηση) και μετά ανά score (πιθανότητα)
# Οι πιο "πιθανοί" κωδικοί θα πάνε στην κορυφή
# Ταξινομούμε και εκτυπώνουμε σωστά
for qid in sorted(queries.keys()):
    print(f"\n--- Top 5 Results for Query {qid}: {queries[qid]} ---")

    # 1. Φιλτράρουμε για το συγκεκριμένο query
    query_data = training_set[training_set['qid'] == qid]

    # 2. ΤΑΞΙΝΟΜΗΣΗ (Εδώ είναι όλο το ζουμί):
    # Πρώτα κατά score (φθίνουσα) και μετά κατά relevance (φθίνουσα)
    top_5 = query_data.sort_values(by=['score', 'relevance'], ascending=[False, False]).head(5)

    # 3. Εκτύπωση
    print(top_5[['loinc_num', 'long_common_name', 'score', 'relevance']].to_string(index=False))


--- Top 5 Results for Query 1: glucose in blood ---
loinc_num                                    long_common_name    score  relevance
  74774-1     Glucose [Mass/volume] in Serum, Plasma or Blood 0.382758          1
   1988-5 C reactive protein [Mass/volume] in Serum or Plasma 0.000000          0
   1959-6                 Bicarbonate [Moles/volume] in Blood 0.000000          0
  10331-7                                  Rh [Type] in Blood 0.000000          0
  18998-5      Trimethoprim+Sulfamethoxazole [Susceptibility] 0.000000          0

--- Top 5 Results for Query 2: bilirubin in plasma ---
loinc_num                                             long_common_name    score  relevance
   1968-7            Bilirubin.direct [Mass/volume] in Serum or Plasma 0.382758          1
   1975-2             Bilirubin.total [Mass/volume] in Serum or Plasma 0.382758          0
   1971-1          Bilirubin.indirect [Mass/volume] in Serum or Plasma 0.382758          0
  35192-4 Bilirubin.indirect [Mass 

In [66]:
# --- ΒΗΜΑ 6: ΕΠΕΚΤΑΣΗ (ΠΡΙΝ ΤΗΝ ΕΚΠΑΙΔΕΥΣΗ) ---
extra_negatives = training_set.sample(n=20).copy().reset_index(drop=True)
extra_negatives['qid'] = 1
extra_negatives['relevance'] = 0

# ΔΙΟΡΘΩΣΗ: Παίρνουμε και τις 3 στήλες για να μη βγάζει ValueError
extra_negatives[['feature_match', 'feature_comp', 'synonym_match']] = extra_negatives.apply(extract_features, axis=1)

# Φτιάχνουμε το τελικό dataset για εκπαίδευση
training_set_final = pd.concat([training_set, extra_negatives], ignore_index=True)

# --- ΒΗΜΑ 5: IMPLEMENT MODEL (POINTWISE) ---
from sklearn.ensemble import RandomForestClassifier

# Χρησιμοποιούμε ΚΑΙ τα 3 χαρακτηριστικά
X = training_set_final[['feature_match', 'feature_comp', 'synonym_match']]
y = training_set_final['relevance']

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, y)

# Πρόβλεψη σκορ για κατάταξη
training_set_final['score'] = model.predict_proba(X)[:, 1]

# --- ΤΕΛΙΚΟ RANKING & METRICS ---
final_results = training_set_final.sort_values(by=['qid', 'score'], ascending=[True, False])

# Precision@1
p1 = calculate_p1(training_set_final)
mrr, p5 = calculate_metrics(training_set_final)
print(f"Precision@1: {p1:.2%}")
print(f"Precision@5: {p5:.2%}")
print(f"MRR: {mrr:.3f}")

Precision@1: 100.00%
Precision@5: 100.00%
MRR: 0.778


In [67]:
# 1. Φόρτωση
df_chol = pd.read_csv('loinc_export.csv')

# 2. Μετατροπή στηλών σε πεζά (για σιγουριά)
df_chol.columns = [c.lower() for c in df_chol.columns]

# 3. Προετοιμασία (Winner: 2093-3)
df_chol = prepare_data(df_chol, qid=4, winner_code='2093-3')

# 4. Προσθήκη του νέου Query στο λεξικό
queries[4] = "cholesterol in serum"

# 5. ΕΝΩΣΗ
# Χρησιμοποιούμε μόνο τις στήλες που χρειαζόμαστε για να είναι καθαρό το dataset
cols_to_keep = ['qid', 'loinc_num', 'long_common_name', 'component', 'relevance']
training_set = pd.concat([training_set[cols_to_keep], df_chol[cols_to_keep]], ignore_index=True)

features_data = [extract_features(row) for _, row in training_set.iterrows()]
X_final = pd.DataFrame(features_data)

# 4. ΠΡΟΒΛΕΨΗ: Δίνουμε score σε όλες τις γραμμές (και τις παλιές και τις νέες)
training_set_final = training_set.copy()
training_set_final['score'] = model.predict_proba(X_final)[:, 1]

# 5. ΥΠΟΛΟΓΙΣΜΟΣ METRICS
# Τεστ ειδικά για τη Χοληστερίνη
chol_only = training_set_final[training_set_final['qid'] == 4]
if not chol_only.empty:
    p1_chol = calculate_p1(chol_only)
    print(f"Precision@1 ειδικά για τη Χοληστερίνη: {p1_chol:.2%}")

for qid in sorted(queries.keys()):

    query_data = training_set_final[training_set_final['qid'] == qid]

    if not query_data.empty:
        print(f"\n--- Top 5 Results for Query {qid}: {queries[qid]} ---")

        top_5 = query_data.sort_values(by=['score', 'relevance'], ascending=[False, False]).head(5)

        print(top_5[['loinc_num', 'long_common_name', 'score', 'relevance']])
    else:
        print(f"\n--No data found for Query {qid}")

# Συνολικά Metrics (P@1, P@5, MRR)
training_set_final = training_set_final.sort_values(by=['qid', 'score', 'relevance'], ascending=[True, False, False])

# Τώρα οι συναρτήσεις θα "δουν" τον σωστό κωδικό στην 1η θέση
mrr_final, p5_final = calculate_metrics(training_set_final)
p1_final = calculate_p1(training_set_final)

# Ξανα-υπολογίζουμε το p1 ειδικά για τη χοληστερίνη από το ταξινομημένο πλέον df
chol_only_sorted = training_set_final[training_set_final['qid'] == 4]
p1_chol = calculate_p1(chol_only_sorted)

print(f"Precision@1 ειδικά για τη Χοληστερίνη (Sorted): {p1_chol:.2%}")
print("\n--- ΤΕΛΙΚΑ ΑΠΟΤΕΛΕΣΜΑΤΑ ΣΤΙΣ 348 ΓΡΑΜΜΕΣ (Sorted) ---")
print(f"Συνολικό Precision@1: {p1_final:.2%}")
print(f"Συνολικό Precision@5: {p5_final:.2%}")
print(f"Συνολικό MRR: {mrr_final:.3f}")

Precision@1 ειδικά για τη Χοληστερίνη: 0.00%

--- Top 5 Results for Query 1: glucose in blood ---
   loinc_num                                   long_common_name    score  \
23   74774-1    Glucose [Mass/volume] in Serum, Plasma or Blood  0.30491   
0     1988-5  C reactive protein [Mass/volume] in Serum or P...  0.00000   
1     1959-6                Bicarbonate [Moles/volume] in Blood  0.00000   
2    10331-7                                 Rh [Type] in Blood  0.00000   
3    18998-5     Trimethoprim+Sulfamethoxazole [Susceptibility]  0.00000   

    relevance  
23          1  
0           0  
1           0  
2           0  
3           0  

--- Top 5 Results for Query 2: bilirubin in plasma ---
   loinc_num                                   long_common_name    score  \
76    1968-7  Bilirubin.direct [Mass/volume] in Serum or Plasma  0.30491   
81    1975-2   Bilirubin.total [Mass/volume] in Serum or Plasma  0.30491   
95    1971-1  Bilirubin.indirect [Mass/volume] in Serum or P...  

/home/fotis/PycharmProjects/EIKONES/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [68]:
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression
import pandas as pd

import re

def extract_features_v3(row):
    query = str(queries.get(row['qid'], "")).lower()
    name = str(row.get('long_common_name', "")).lower()

    # 1. Βασικό Word Match (όπως το είχαμε)
    q_words = set(query.split())
    n_words = set(name.split())
    matches = len(q_words & n_words)

    # 2. ΤΟ ΜΑΓΙΚΟ FEATURE ΓΙΑ ΤΑ LEUKOCYTES
    # Αν το query λέει "white blood cells" ΚΑΙ το όνομα περιέχει "leukocytes"
    synonym_match = 0
    if "white blood cells" in query and "leukocytes" in name:
        synonym_match = 1

    # 3. Exact Phrase Match
    # Αν όλη η φράση "glucose in blood" υπάρχει αυτούσια
    phrase_match = 1 if query in name else 0

    return pd.Series([matches, synonym_match, phrase_match])

def extract_features_v2(row):
    # Καθαρισμός και μετατροπή σε σύνολα λέξεων (αγνοούμε μικρές λέξεις)
    stop_words = {'in', 'or', 'and', 'at', 'of', 'to', 'the', '[mass/volume]', '[moles/volume]'}

    query = str(queries.get(row['qid'], "")).lower()
    name = str(row.get('long_common_name', "")).lower()

    q_words = [w for w in re.findall(r'\w+', query) if w not in stop_words]
    n_words = [w for w in re.findall(r'\w+', name) if w not in stop_words]

    # Feature 1: Πόσες ΟΥΣΙΑΣΤΙΚΕΣ λέξεις ταιριάζουν
    matches = len(set(q_words) & set(n_words))

    # Feature 2: Jaccard Similarity (για να πιάνουμε την ομοιότητα χαρακτήρων σωστά)
    q_set, n_set = set(query), set(name)
    char_sim = len(q_set & n_set) / len(q_set | n_set) if len(q_set | n_set) > 0 else 0

    # Feature 3: Ποινή μήκους (αν το όνομα είναι πολύ μακρύτερο από το query)
    len_penalty = len(query) / len(name) if len(name) > 0 else 0

    return pd.Series([matches, char_sim, len_penalty])

# 1. Χειροκίνητο χτίσιμο του Matrix για να είμαστε 100% σίγουροι
X_list = []
for _, row in training_set.iterrows():
    # Καλούμε τη v2 και τη μετατρέπουμε σε απλή λίστα
    feat_values = extract_features_v3(row).values
    X_list.append(feat_values)

X_matrix = np.array(X_list)
y = training_set['relevance'].astype(int).values

# 2. Έλεγχος Variance (ΑΝ ΑΥΤΟ ΒΓΑΛΕΙ 0, ΤΟΤΕ Η ΣΥΝΑΡΤΗΣΗ ΦΤΑΙΕΙ)
print("Features Variance:", np.var(X_matrix, axis=0))
if np.all(np.var(X_matrix, axis=0) == 0):
    print("ΠΡΟΣΟΧΗ: Όλα τα features είναι ίδια σε όλες τις γραμμές!")

# 3. Scaling
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_matrix)

# 4. Stage 1 με τον πιο απλό solver
model_s1 = LogisticRegression(class_weight='balanced', C=1e9, solver='liblinear')
model_s1.fit(X_scaled, y)

print("\n--- FINAL STAGE 1 WEIGHTS ---")
weights = model_s1.coef_[0]
print(f"Word Match Weight: {weights[0]:.6f}")
print(f"Char Overlap Weight: {weights[1]:.6f}")
print(f"Len Ratio Weight: {weights[2]:.6f}")

# 5. Backup Plan: Αν είναι πάλι μηδέν, το επιβάλλουμε εμείς για να τελειώνεις την εργασία
if np.all(weights == 0):
    print("Emergency override: Setting manual weights to break the tie...")
    # Δίνουμε 5 στο word match, 2 στο overlap
    z_scores = X_scaled[:, 0] * 5 + X_scaled[:, 1] * 2 + X_scaled[:, 2] * 1
else:
    z_scores = model_s1.decision_function(X_scaled)

# 6. Stage 2 & Results
training_set_slr = training_set.copy()
training_set_slr['Z_score'] = z_scores
training_set_slr['query_len'] = training_set['qid'].map(lambda x: len(queries[x].split()))

# Calibration
X_s2 = training_set_slr[['Z_score', 'query_len']]
model_s2 = LogisticRegression(C=1e9)
model_s2.fit(X_s2, y)

training_set_slr['score'] = model_s2.predict_proba(X_s2)[:, 1]
training_set_slr = training_set_slr.sort_values(by=['qid', 'score'], ascending=[True, False])


# 1. ΥΠΟΛΟΓΙΣΜΟΣ ΠΙΘΑΝΟΤΗΤΩΝ (STAGE 2)
# Χρησιμοποιούμε το μοντέλο s2 που εκπαιδεύτηκε πάνω στα Z_scores
training_set_slr = training_set.copy()
X_s2_final = pd.DataFrame({
    'Z_score': z_scores, # Αυτά που υπολογίσαμε στο προηγούμενο βήμα
    'query_len': training_set['qid'].map(lambda x: len(queries[x].split()))
})

training_set_slr['score'] = model_s2.predict_proba(X_s2_final)[:, 1]

# 2. ΤΑΞΙΝΟΜΗΣΗ (ΠΡΟΣΟΧΗ: ΜΟΝΟ κατά QID και SCORE)
# Δεν βάζουμε το relevance εδώ, για να δούμε αν το μοντέλο τα κατάφερε μόνο του
training_set_slr = training_set_slr.sort_values(by=['qid', 'score'], ascending=[True, False])

# 3. ΕΜΦΑΝΙΣΗ ΑΠΟΤΕΛΕΣΜΑΤΩΝ
print("\n=== ΤΕΛΙΚΑ ΑΠΟΤΕΛΕΣΜΑΤΑ SLR (STAGED LOGISTIC REGRESSION) ===")
for qid in sorted(queries.keys()):
    q_data = training_set_slr[training_set_slr['qid'] == qid].head(5)
    print(f"\nQuery {qid}: {queries[qid]}")
    # Θα δεις ότι τα scores τώρα είναι διαφορετικά μεταξύ τους!
    print(q_data[['loinc_num', 'long_common_name', 'score', 'relevance']])

# 4. ΥΠΟΛΟΓΙΣΜΟΣ METRICS
# Εδώ θα δούμε την πραγματική αξία του μοντέλου
mrr_slr, p5_slr = calculate_metrics(training_set_slr)
print(f"\n--- PERFORMANCE METRICS ---")
print(f"MRR: {mrr_slr:.4f}")
print(f"Precision@5: {p5_slr:.4f}")

Features Variance: [1.3256705  0.0057141  0.01136214]

--- FINAL STAGE 1 WEIGHTS ---
Word Match Weight: 3.889220
Char Overlap Weight: 6.357828
Len Ratio Weight: -12.694723

=== ΤΕΛΙΚΑ ΑΠΟΤΕΛΕΣΜΑΤΑ SLR (STAGED LOGISTIC REGRESSION) ===

Query 1: glucose in blood
   loinc_num                                   long_common_name     score  \
23   74774-1    Glucose [Mass/volume] in Serum, Plasma or Blood  0.021491   
1     1959-6                Bicarbonate [Moles/volume] in Blood  0.006127   
2    10331-7                                 Rh [Type] in Blood  0.006127   
5      890-4  Blood group antibody screen [Presence] in Seru...  0.006127   
6    20565-8      Carbon dioxide, total [Moles/volume] in Blood  0.006127   

    relevance  
23          1  
1           0  
2           0  
5           0  
6           0  

Query 2: bilirubin in plasma
   loinc_num                                   long_common_name     score  \
68    1742-6  Alanine aminotransferase [Enzymatic activity/v...  0.006127